# Medical Insurance Charges Prediction

## Project Overview

This project investigates the factors associated with individual medical insurance charges and develops regression models to predict insurance costs.

The analysis focuses on practical insurance analytics using **R**, including exploratory data analysis, interaction effects, model validation, diagnostics and prediction.

**Business question:** Can demographic, lifestyle and health-related characteristics be used to predict individual medical insurance charges?

## 1. Dataset

The dataset contains 1,338 observations and 7 variables:

- `age`
- `sex`
- `bmi`
- `children`
- `smoker`
- `region`
- `charges`

The target variable is `charges`, representing individual medical insurance costs.

In [ ]:
library(ggplot2)

kaggle_path <- "../input/datasets/mosapabdelghany/medical-insurance-cost-dataset/insurance.csv"
local_path  <- "data/insurance.csv"

if (file.exists(kaggle_path)) {
  insurance <- read.csv(kaggle_path)
} else if (file.exists(local_path)) {
  insurance <- read.csv(local_path)
} else if (file.exists("insurance.csv")) {
  insurance <- read.csv("insurance.csv")
} else {
  stop("insurance.csv was not found. Download the Kaggle dataset and place it in data/insurance.csv.")
}

insurance[] <- lapply(
  insurance,
  function(x) if (is.character(x)) trimws(x) else x
)

insurance$sex <- factor(insurance$sex)
insurance$smoker <- factor(insurance$smoker)
insurance$region <- factor(insurance$region)

head(insurance)

## 2. Data Quality Check

Before modelling, the dataset is checked for structure, summary statistics and missing values.

No missing-value treatment is required if the dataset returns zero missing observations for every variable.

In [ ]:
str(insurance)
summary(insurance)

missing_values <- colSums(is.na(insurance))
missing_values

cat("Rows:", nrow(insurance), "\n")
cat("Columns:", ncol(insurance), "\n")

## 3. Exploratory Data Analysis

The exploratory analysis focuses on the main variables that may help explain variation in medical insurance charges.

In the original analysis, smoking status showed a particularly large difference in average charges between smokers and non-smokers. This makes smoking status an important candidate predictor.

In [ ]:
aggregate(charges ~ smoker, data = insurance, FUN = mean)

In [ ]:
ggplot(insurance, aes(x = smoker)) +
  geom_bar() +
  labs(
    title = "Number of Policyholders by Smoking Status",
    x = "Smoking Status",
    y = "Number of Policyholders"
  ) +
  theme_minimal()

In [ ]:
ggplot(insurance, aes(x = smoker, y = charges)) +
  geom_boxplot() +
  labs(
    title = "Medical Insurance Charges by Smoking Status",
    x = "Smoking Status",
    y = "Medical Insurance Charges"
  ) +
  theme_minimal()

### Age and Charges

Age is examined because the exploratory analysis indicates a positive relationship between age and medical insurance charges.

In [ ]:
ggplot(insurance, aes(x = age, y = charges)) +
  geom_point(alpha = 0.6) +
  labs(
    title = "Age vs Medical Insurance Charges",
    x = "Age",
    y = "Medical Insurance Charges"
  ) +
  theme_minimal()

### BMI and Charges

BMI also shows a positive relationship with charges. The pattern differs across smoking groups, motivating the use of an interaction term between BMI and smoking status in one candidate model.

In [ ]:
ggplot(insurance, aes(x = bmi, y = charges, shape = smoker)) +
  geom_point(alpha = 0.6) +
  labs(
    title = "BMI vs Medical Insurance Charges",
    x = "BMI",
    y = "Medical Insurance Charges",
    shape = "Smoker"
  ) +
  theme_minimal()

In [ ]:
ggplot(insurance, aes(x = factor(children), y = charges)) +
  geom_boxplot() +
  labs(
    title = "Medical Insurance Charges by Number of Children",
    x = "Number of Children",
    y = "Medical Insurance Charges"
  ) +
  theme_minimal()

In [ ]:
ggplot(insurance, aes(x = region, y = charges)) +
  geom_boxplot() +
  labs(
    title = "Medical Insurance Charges by Region",
    x = "Region",
    y = "Medical Insurance Charges"
  ) +
  theme_minimal()

## 4. Correlation Analysis

Correlations are calculated for the numerical variables.

Correlation is useful for identifying linear relationships, but it does not by itself establish causation and does not capture relationships involving categorical variables.

In [ ]:
cor(
  insurance[c("age", "bmi", "children", "charges")]
)

## 5. Train-Test Split

The data is divided into 80% training observations and 20% testing observations.

The training data is used for model development. The test data is kept separate until the final evaluation so that predictive performance can be assessed on previously unseen observations.

In [ ]:
set.seed(123)

train_index <- sample(
  seq_len(nrow(insurance)),
  size = floor(0.80 * nrow(insurance))
)

train_data <- insurance[train_index, ]
test_data  <- insurance[-train_index, ]

cat("Training observations:", nrow(train_data), "\n")
cat("Testing observations:", nrow(test_data), "\n")

## 6. Candidate Regression Models

Four multiple linear regression specifications are considered.

**Model 1 — Main effects**

`charges ~ age + sex + bmi + children + smoker + region`

**Model 2 — BMI × smoking interaction**

`charges ~ age + sex + bmi + children + smoker * bmi + region`

**Model 3 — Age × smoking interaction**

`charges ~ age * smoker + sex + bmi + children + region`

**Model 4 — Reduced BMI × smoking interaction model**

`charges ~ age + children + smoker * bmi + region`

The interaction terms allow the effect of one variable to depend on another variable.

In [ ]:
model_formulas <- list(
  Model_1 = charges ~ age + sex + bmi + children + smoker + region,
  Model_2 = charges ~ age + sex + bmi + children + smoker * bmi + region,
  Model_3 = charges ~ age * smoker + sex + bmi + children + region,
  Model_4 = charges ~ age + children + smoker * bmi + region
)

## 7. Five-Fold Cross-Validation

The candidate models are compared using 5-fold cross-validation **within the training data**.

This is an important improvement over comparing models that were trained on the entire dataset and then evaluated on the same observations.

The test set remains untouched during model selection.

In [ ]:
set.seed(123)

k <- 5
fold_id <- sample(
  rep(seq_len(k), length.out = nrow(train_data))
)

cv_evaluate <- function(formula, data, folds) {

  fold_metrics <- lapply(seq_len(max(folds)), function(i) {

    fold_train <- data[folds != i, ]
    fold_valid <- data[folds == i, ]

    fit <- lm(formula, data = fold_train)
    pred <- predict(fit, newdata = fold_valid)
    actual <- fold_valid$charges

    data.frame(
      RMSE = sqrt(mean((actual - pred)^2)),
      MAE = mean(abs(actual - pred))
    )
  })

  fold_metrics <- do.call(rbind, fold_metrics)

  data.frame(
    CV_RMSE = mean(fold_metrics$RMSE),
    CV_MAE = mean(fold_metrics$MAE)
  )
}

cv_results <- do.call(
  rbind,
  lapply(
    model_formulas,
    cv_evaluate,
    data = train_data,
    folds = fold_id
  )
)

cv_results$Model <- rownames(cv_results)
rownames(cv_results) <- NULL

cv_results <- cv_results[
  order(cv_results$CV_RMSE),
  c("Model", "CV_RMSE", "CV_MAE")
]

cv_results

## 8. Select the Final Model

The candidate model with the lowest cross-validation RMSE is selected.

The selected specification is then refitted using all observations in the training set.

In [ ]:
best_model_name <- cv_results$Model[1]
best_formula <- model_formulas[[best_model_name]]

cat("Selected model:", best_model_name, "\n")
best_formula

final_model <- lm(best_formula, data = train_data)

summary(final_model)
confint(final_model)

## 9. Final Test-Set Evaluation

The selected model is evaluated once on the untouched test set.

**Metrics:**

- **R²:** proportion of variation in charges explained by the model.
- **RMSE:** penalizes larger prediction errors more heavily.
- **MAE:** average absolute prediction error.

In [ ]:
test_predictions <- predict(final_model, newdata = test_data)
actual_charges <- test_data$charges

RMSE <- sqrt(mean((actual_charges - test_predictions)^2))
MAE <- mean(abs(actual_charges - test_predictions))

SS_res <- sum((actual_charges - test_predictions)^2)
SS_tot <- sum((actual_charges - mean(actual_charges))^2)

Test_R2 <- 1 - SS_res / SS_tot

final_results <- data.frame(
  Model = best_model_name,
  Test_R2 = Test_R2,
  RMSE = RMSE,
  MAE = MAE
)

final_results

## 10. Actual vs Predicted Charges

A well-performing model should generally produce predictions close to the 45-degree reference line.

Large deviations from the line indicate observations for which the model has larger prediction errors.

In [ ]:
prediction_plot_data <- data.frame(
  Actual = actual_charges,
  Predicted = test_predictions
)

ggplot(prediction_plot_data, aes(x = Actual, y = Predicted)) +
  geom_point(alpha = 0.6) +
  geom_abline(
    slope = 1,
    intercept = 0,
    linetype = "dashed"
  ) +
  labs(
    title = paste("Actual vs Predicted Charges -", best_model_name),
    x = "Actual Charges",
    y = "Predicted Charges"
  ) +
  theme_minimal()

## 11. Regression Diagnostics

The diagnostic plots are used to assess whether the assumptions of the linear regression model are reasonably satisfied.

The four plots examine:

1. **Residuals vs Fitted** — possible non-linearity.
2. **Normal Q-Q** — approximate normality of residuals.
3. **Scale-Location** — whether residual variance is reasonably constant.
4. **Residuals vs Leverage** — potentially influential observations.

In [ ]:
par(mfrow = c(2, 2))
plot(final_model)
par(mfrow = c(1, 1))

## 12. Example New-Customer Prediction

The final model can be used to estimate charges for a new policyholder with specified characteristics.

The prediction interval gives a range that reflects uncertainty around an individual prediction.

In [ ]:
new_customer <- data.frame(
  age = 30,
  sex = factor("male", levels = levels(insurance$sex)),
  bmi = 25,
  children = 2,
  smoker = factor("no", levels = levels(insurance$smoker)),
  region = factor("southeast", levels = levels(insurance$region))
)

predict(
  final_model,
  newdata = new_customer,
  interval = "prediction"
)

## 13. Conclusion

This project demonstrates an end-to-end insurance analytics workflow in R.

The analysis:

- explored the main drivers of medical insurance charges;
- identified smoking status as an especially important risk factor;
- tested interaction effects between smoking status and BMI or age;
- compared alternative regression specifications using cross-validation;
- evaluated the selected model on unseen test observations;
- assessed regression assumptions using diagnostic plots; and
- demonstrated prediction for a new customer.

The final numerical model performance is generated directly by the notebook after execution, rather than being hard-coded into the report.

## 14. Limitations and Future Work

This dataset is useful for demonstrating insurance analytics techniques, but it is not a complete real-world pricing dataset.

Important limitations include the absence of policy exposure, claim frequency, claim severity, policy limits, policy dates and richer underwriting variables.

Future work could include:

- generalized linear models for insurance pricing;
- log-transformed or other skew-aware models;
- non-linear age and BMI effects;
- regularized regression;
- gradient boosting;
- claim frequency and severity modelling;
- SQL-based insurance portfolio analysis; and
- Power BI dashboard development.